# Experiment 1.2b-i: Lyapunov Exponent of the Gauge Direction Under Each Optimizer

---

## Scientific Motivation

In the theory of **Muon as a Renormalization Group (RG) gauge-fixing mechanism**, weight matrices in deep linear networks possess a fundamental redundancy: any transformation $W \to W \cdot P$ where $P$ is a positive semi-definite (PSD) matrix close to identity does not change the *effective function* computed by the network (up to reparametrization). These are the **gauge directions** in weight space -- analogous to gauge redundancies in field theory.

The central claim of the RG gauge-fixing hypothesis is that **Muon's Newton-Schulz orthogonalization acts as a gauge-fixing mechanism**, actively contracting perturbations along these redundant PSD directions. In contrast, standard optimizers like SGD have no such mechanism and should treat gauge directions neutrally.

## Hypothesis

We test this by measuring the **Lyapunov exponent** $\lambda$ along the gauge direction for each optimizer. The Lyapunov exponent quantifies the exponential rate of divergence (or convergence) of nearby trajectories in dynamical systems:

$$\lambda = \frac{1}{N} \ln \frac{d(N)}{d(0)}$$

where $d(t) = \|W_t' - W_t\|_F$ is the Frobenius distance between an unperturbed trajectory $\{W_t\}$ and a trajectory $\{W_t'\}$ initialized with a small gauge perturbation $W_0' = W_0 \cdot (I + \varepsilon S / \|S\|_F)$, with $S$ symmetric.

**Predictions:**

| Optimizer | Gauge $\lambda$ | Physical $\lambda$ | Interpretation |
|-----------|-----------------|---------------------|----------------|
| **Muon**  | $\lambda \ll 0$ (strongly negative) | $\lambda \approx 0$ | Muon *specifically* contracts gauge directions |
| **SGD**   | $\lambda \approx 0$ (neutral) | $\lambda \approx 0$ | SGD has no gauge-fixing mechanism |
| **Adam**  | $\lambda$ slightly negative | $\lambda \approx 0$ | Adam's second-moment may weakly contract gauge |

**Critical context:**
- The D-TEST (Exp 1.1) confirmed that SGD's disadvantage compounds exponentially with depth (per-layer Lyapunov ~0.095)
- The P17 dynamical systems perspective predicts: SGD trajectories diverge ($\lambda > 0$), Muon converges ($\lambda < 0$)
- This experiment directly measures the Lyapunov exponent predicted by the dynamical systems model

## Methodology Overview

1. **Setup:** A 4-layer deep linear network with $32 \times 32$ weight matrices and quadratic loss
2. **Perturbation:** For each trial, perturb the initial weights in either the *gauge* (symmetric/PSD) or *physical* (skew-symmetric/orthogonal tangent) direction
3. **Evolution:** Run both the base and perturbed trajectories for $N = 200$ optimizer steps
4. **Measurement:** Compute $d(t)$ at each step and extract the Lyapunov exponent $\lambda$
5. **Statistics:** Repeat over 20 random perturbation directions for robust estimation

## Expected Outcomes

- If Muon is a gauge-fixing mechanism: $\lambda_{\text{gauge}}^{\text{Muon}} \ll 0$ (gauge perturbations decay exponentially)
- If gauge-fixing is *specific* to gauge directions: $\lambda_{\text{gauge}}^{\text{Muon}} < \lambda_{\text{physical}}^{\text{Muon}}$
- If SGD lacks gauge-fixing: $\lambda_{\text{gauge}}^{\text{SGD}} \approx 0$ (neutral random walk in gauge space)

---

## 0. Preamble: Experiment Summary (Code Reference)

The cell below was the original experiment description embedded as a Python docstring. It is preserved here as a markdown reference for quick scanning.

**Key parameters:** 4-layer deep linear net, 32x32 matrices, 200 optimizer steps, $\varepsilon = 0.001$, 20 perturbation directions.

**Two perturbation types tested:**
- **Gauge (symmetric/PSD):** $W_0' = W_0 \cdot (I + \varepsilon \cdot S / \|S\|_F)$ where $S = S^T$ -- this probes the redundant gauge direction
- **Physical (skew-symmetric/tangent):** $W_0' = W_0 \cdot \exp(\varepsilon \cdot A / \|A\|_F)$ where $A = -A^T$ -- this probes directions that change the network's function

## 1. Imports and Reproducibility

We use only NumPy for all computations -- no deep learning framework is needed since the experiment involves deep *linear* networks where all operations are matrix multiplications. This keeps the experiment transparent and fully deterministic.

In [ ]:
import numpy as np
import os

print(f"NumPy version: {np.__version__}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
np.random.seed(42)
print("Global random seed set to 42 for full reproducibility.")

## 2. Configuration

The experiment parameters are chosen to balance sensitivity with stability:

- **DIM = 32:** Matrix dimension. Large enough to have a rich spectrum, small enough for fast computation. The gauge space has $32 \times 33 / 2 = 528$ symmetric degrees of freedom.
- **NUM_LAYERS = 4:** Depth of the linear network. The D-TEST showed gauge effects compound exponentially with depth, so 4 layers gives a strong signal.
- **NUM_STEPS = 200:** Trajectory length. Must be long enough for the Lyapunov exponent to converge but short enough to avoid saturation effects.
- **EPSILON = 0.001:** Perturbation magnitude. Small enough to be in the linear response regime (so the Lyapunov exponent is well-defined) but large enough to be above numerical noise.
- **NUM_PERTURBATIONS = 20:** Number of random perturbation directions. Averaging over these gives a robust estimate of the *maximal* Lyapunov exponent along gauge directions.
- **NS_ITERS = 5:** Newton-Schulz iterations for Muon's orthogonalization. 5 iterations gives excellent convergence to the polar factor for matrices near identity.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 200
BATCH_SIZE = 64
LR_MUON = 0.005
MOMENTUM = 0.9
NS_ITERS = 5
EPSILON = 0.001
NUM_PERTURBATIONS = 20

print("=== Experiment Configuration ===")
print(f"  Matrix dimension:        {DIM}x{DIM}")
print(f"  Number of layers:        {NUM_LAYERS}")
print(f"  Total weight parameters: {NUM_LAYERS * DIM * DIM} ({NUM_LAYERS} x {DIM}x{DIM})")
print(f"  Gauge DOF per layer:     {DIM * (DIM + 1) // 2} (symmetric matrices)")
print(f"  Physical DOF per layer:  {DIM * (DIM - 1) // 2} (skew-symmetric matrices)")
print(f"  Optimizer steps:         {NUM_STEPS}")
print(f"  Batch size:              {BATCH_SIZE}")
print(f"  Muon learning rate:      {LR_MUON}")
print(f"  Momentum:                {MOMENTUM}")
print(f"  Newton-Schulz iters:     {NS_ITERS}")
print(f"  Perturbation epsilon:    {EPSILON}")
print(f"  Number of perturbations: {NUM_PERTURBATIONS}")

In [ ]:
# Random target matrix (fixed)
W_target = np.random.randn(DIM, DIM) * 0.5

print(f"Target matrix W_target: shape={W_target.shape}, dtype={W_target.dtype}")
print(f"  Frobenius norm:   {np.linalg.norm(W_target, 'fro'):.4f}")
print(f"  Spectral norm:    {np.linalg.norm(W_target, ord=2):.4f}")
print(f"  Condition number: {np.linalg.cond(W_target):.4f}")
svs = np.linalg.svd(W_target, compute_uv=False)
print(f"  Singular values:  min={svs.min():.4f}, max={svs.max():.4f}, mean={svs.mean():.4f}")

In [ ]:
# Random input data (fixed batch)
X_data = np.random.randn(DIM, BATCH_SIZE) * 0.3

print(f"Input data X_data: shape={X_data.shape} (dim x batch_size)")
print(f"  Frobenius norm: {np.linalg.norm(X_data, 'fro'):.4f}")
print(f"  Per-sample mean norm: {np.mean(np.linalg.norm(X_data, axis=0)):.4f}")

In [ ]:
# Output directory
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Running in a notebook -- use current working directory
    SCRIPT_DIR = os.getcwd()

print(f"Output directory: {SCRIPT_DIR}")

## 3. Core Functions: Network, Loss, and Gradient Computation

### Deep Linear Network Architecture

The network computes $f(X) = W_L \cdot W_{L-1} \cdots W_1 \cdot X$. Despite being "deep," the function is linear -- the product $W_{\text{eff}} = W_L \cdots W_1$ is a single matrix. The depth creates a **redundancy**: for any invertible $P$, replacing $W_i \to W_i P$ and $W_{i+1} \to P^{-1} W_{i+1}$ preserves $W_{\text{eff}}$. When $P$ is close to identity and positive definite ($P = I + \varepsilon S$ with $S$ symmetric), this is a **gauge transformation**.

### Weight Initialization

Weights are initialized near identity ($W = I + 0.1 \cdot \text{noise}$) to ensure:
1. The initial effective matrix $W_{\text{eff}} \approx I$ is well-conditioned
2. The optimization problem is non-trivial but solvable
3. The initial perturbation $\varepsilon = 0.001$ is truly in the linear regime

### Loss Function

$\mathcal{L} = \frac{1}{2N} \|W_{\text{eff}} X - W_{\text{target}} X\|_F^2$

This is a quadratic loss in the weight matrices, ensuring smooth gradients and no local minima (the landscape is convex in $W_{\text{eff}}$, though not in the individual $W_i$).

In [ ]:
def init_weights(num_layers, seed=42):
    """Initialize layers near identity for stability."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
def forward(weights, X):
    """Forward pass: W_L @ ... @ W_1 @ X."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, target):
    """Loss = 0.5 * ||W_product @ X - T @ X||^2 / N."""
    pred = forward(weights, X)
    target_out = target @ X
    diff = pred - target_out
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

In [ ]:
def compute_gradients(weights, X, target):
    """Backprop through deep linear net."""
    num_layers = len(weights)
    N = X.shape[1]

    # Forward pass storing activations
    activations = [X.copy()]
    out = X.copy()
    for W in weights:
        out = W @ out
        activations.append(out.copy())

    # Backward pass
    target_out = target @ X
    delta = (activations[-1] - target_out) / N

    grads = []
    for i in range(num_layers - 1, -1, -1):
        G = delta @ activations[i].T
        grads.insert(0, G)
        if i > 0:
            delta = weights[i].T @ delta

    return grads

### Newton-Schulz Orthogonalization (The Heart of Muon)

This is the key function that distinguishes Muon from SGD. Given a gradient matrix $G$, the Newton-Schulz iteration computes its **orthogonal polar factor** -- the closest orthogonal matrix $U V^T$ from the SVD $G = U \Sigma V^T$.

The iteration is: $X_{k+1} = \frac{3}{2} X_k - \frac{1}{2} X_k (X_k^T X_k)$, starting from $X_0 = G / \|G\|_F$.

**Why this matters for gauge-fixing:** The polar factor projects out all singular value information, keeping only the rotation. Since gauge transformations act on singular values (they are symmetric/PSD), replacing the gradient with its polar factor removes the component that would "push" along gauge directions. This is exactly the mechanism by which Muon is predicted to fix the gauge.

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=NS_ITERS):
    """
    Newton-Schulz iteration to approximate the orthogonal polar factor.
    Returns closest orthogonal matrix to G (i.e., U @ V^T from SVD).
    """
    norm = np.linalg.norm(G, ord='fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A

    return X

### Learning Rate Calibration for Fair Comparison

A critical methodological concern: if we use a much larger learning rate for one optimizer, the results could be confounded by the learning rate rather than the optimizer's inherent dynamics. We find the maximum stable learning rate for SGD, ensuring both optimizers are operating in their natural regime. Muon's learning rate is fixed at $0.005$.

In [ ]:
def find_stable_lr_sgd():
    """Find maximum stable SGD learning rate."""
    candidates = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]
    for lr in candidates:
        np.random.seed(42)
        weights = init_weights(NUM_LAYERS)
        velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
        initial_loss = compute_loss(weights, X_data, W_target)
        stable = True
        for step in range(80):
            grads = compute_gradients(weights, X_data, W_target)
            for i in range(NUM_LAYERS):
                velocities[i] = MOMENTUM * velocities[i] + grads[i]
                weights[i] -= lr * velocities[i]
            loss = compute_loss(weights, X_data, W_target)
            if np.isnan(loss) or loss > initial_loss * 50:
                stable = False
                break
        if stable:
            return lr
    return 0.001

### Perturbation Generators: Gauge vs. Physical Directions

The mathematical distinction between the two perturbation types is crucial:

- **Symmetric matrices** ($S = S^T$) generate the **gauge directions**. The perturbation $W \to W(I + \varepsilon S)$ modifies the singular values of $W$ without rotating the singular vectors. This is the redundancy that gauge-fixing should eliminate.

- **Skew-symmetric matrices** ($A = -A^T$) generate the **physical directions**. The exponential $\exp(\varepsilon A)$ is an orthogonal matrix, so $W \to W \exp(\varepsilon A)$ rotates $W$ without changing its singular values. This direction changes the actual function computed by the network and should *not* be suppressed by a good optimizer.

In [ ]:
def random_symmetric(dim, rng):
    """Generate a random symmetric matrix."""
    A = rng.randn(dim, dim)
    return (A + A.T) / 2.0

In [ ]:
def random_skew_symmetric(dim, rng):
    """Generate a random skew-symmetric matrix."""
    A = rng.randn(dim, dim)
    return (A - A.T) / 2.0

### Matrix Exponential via Pade Approximation

For physical perturbations, we need $\exp(\varepsilon A)$ where $A$ is skew-symmetric. The matrix exponential of a skew-symmetric matrix is guaranteed to be orthogonal ($\exp(A)^T = \exp(A^T) = \exp(-A) = \exp(A)^{-1}$), so the perturbation $W \to W \exp(\varepsilon A)$ stays on the Stiefel manifold if $W$ was orthogonal. We use scaling-and-squaring with a Pade approximant for numerical stability.

In [ ]:
def matrix_exponential_pade(A, order=6):
    """
    Compute matrix exponential via scaling-and-squaring with Pade approximation.
    For small ||A||, direct Pade is accurate. For larger ||A||, we scale down first.
    """
    norm_A = np.linalg.norm(A, ord='fro')
    if norm_A < 1e-15:
        return np.eye(A.shape[0])

    # Scaling: find s such that ||A/2^s|| < 1
    s = max(0, int(np.ceil(np.log2(norm_A + 1e-15))))
    A_scaled = A / (2 ** s)

    # Pade [order/order] approximant
    I = np.eye(A.shape[0])
    N_pade = I.copy()
    D_pade = I.copy()
    A_power = I.copy()
    c = 1.0

    for k in range(1, order + 1):
        c *= (order - k + 1) / (k * (2 * order - k + 1))
        A_power = A_power @ A_scaled
        N_pade += c * A_power
        D_pade += ((-1) ** k) * c * A_power

    # expm(A_scaled) ~ D^{-1} N
    result = np.linalg.solve(D_pade, N_pade)

    # Squaring phase
    for _ in range(s):
        result = result @ result

    return result

## 4. Optimizer Step Functions

### SGD with Momentum
Standard gradient descent with Nesterov-style momentum: $v_{t+1} = \mu v_t + g_t$, $W_{t+1} = W_t - \eta v_{t+1}$.

The gradient $g_t$ contains components in both gauge and physical directions. SGD applies them all equally -- it has no mechanism to distinguish gauge from physical.

### Muon with Momentum
Same momentum scheme, but the gradient is first **orthogonalized** via Newton-Schulz: $\hat{g}_t = \text{NS}(g_t)$.

The orthogonalized gradient $\hat{g}_t$ is an orthogonal matrix (or close to one). Orthogonal matrices have all singular values equal to 1 -- they carry zero information about the gauge direction. This is the theoretical basis for predicting $\lambda_{\text{gauge}}^{\text{Muon}} < 0$.

In [ ]:
def sgd_step(weights, velocities, lr):
    """One step of SGD with momentum."""
    grads = compute_gradients(weights, X_data, W_target)
    for i in range(len(weights)):
        velocities[i] = MOMENTUM * velocities[i] + grads[i]
        weights[i] = weights[i] - lr * velocities[i]
    return weights, velocities

In [ ]:
def muon_step(weights, velocities, lr):
    """One step of Muon with momentum."""
    grads = compute_gradients(weights, X_data, W_target)
    for i in range(len(weights)):
        ortho_grad = newton_schulz_orthogonalize(grads[i])
        velocities[i] = MOMENTUM * velocities[i] + ortho_grad
        weights[i] = weights[i] - lr * velocities[i]
    return weights, velocities

## 5. Lyapunov Measurement Engine

This is the core measurement apparatus. The procedure for each trial is:

1. **Initialize** base weights $\{W_i^{(0)}\}$ (same for all trials)
2. **Perturb** to get $\{W_i'^{(0)}\}$ using a random gauge or physical perturbation
3. **Run** both base and perturbed trajectories for $N$ steps under the same optimizer
4. **Track** $d(t) = \sqrt{\sum_i \|W_i^{(t)} - W_i'^{(t)}\|_F^2}$ at each step
5. **Compute** $\lambda = \frac{1}{N} \ln \frac{d(N)}{d(0)}$

The Lyapunov exponent $\lambda$ measures the exponential rate of divergence/convergence:
- $\lambda < 0$: perturbation decays as $d(t) \sim d(0) e^{\lambda t}$ -- the optimizer **stabilizes** this direction
- $\lambda = 0$: perturbation neither grows nor decays -- the optimizer is **neutral**
- $\lambda > 0$: perturbation grows exponentially -- the optimizer is **unstable** in this direction

In [ ]:
def run_trajectory(weights_init, optimizer, lr, num_steps):
    """
    Run optimizer for num_steps from weights_init.
    Returns list of weight snapshots at each step (including step 0).
    """
    weights = [w.copy() for w in weights_init]
    velocities = [np.zeros_like(w) for w in weights]

    trajectory = [[w.copy() for w in weights]]

    for step in range(num_steps):
        if optimizer == 'sgd':
            weights, velocities = sgd_step(weights, velocities, lr)
        elif optimizer == 'muon':
            weights, velocities = muon_step(weights, velocities, lr)
        else:
            raise ValueError(f"Unknown optimizer: {optimizer}")

        trajectory.append([w.copy() for w in weights])

        # Check for divergence
        loss = compute_loss(weights, X_data, W_target)
        if np.isnan(loss) or loss > 1e10:
            # Pad remaining with last valid state
            for _ in range(num_steps - step - 1):
                trajectory.append([w.copy() for w in weights])
            break

    return trajectory

In [ ]:
def compute_trajectory_distance(traj_a, traj_b):
    """
    Compute d(t) = ||W_t^a - W_t^b||_F (summed over all layers) at each timestep.
    """
    num_steps = min(len(traj_a), len(traj_b))
    distances = []
    for t in range(num_steps):
        d = 0.0
        for i in range(len(traj_a[t])):
            d += np.linalg.norm(traj_a[t][i] - traj_b[t][i], 'fro') ** 2
        distances.append(np.sqrt(d))
    return np.array(distances)

In [ ]:
def compute_per_layer_distances(traj_a, traj_b):
    """Compute per-layer distances over time."""
    num_steps = min(len(traj_a), len(traj_b))
    num_layers = len(traj_a[0])
    per_layer = np.zeros((num_layers, num_steps))
    for t in range(num_steps):
        for i in range(num_layers):
            per_layer[i, t] = np.linalg.norm(traj_a[t][i] - traj_b[t][i], 'fro')
    return per_layer

In [ ]:
def measure_lyapunov(optimizer, lr, perturbation_type, num_perturbations, seed_base=100):
    """
    Measure Lyapunov exponent for a given optimizer and perturbation type.

    perturbation_type: 'gauge' (symmetric/PSD) or 'physical' (skew-symmetric/tangent)

    Returns:
      lyapunov_exponents: array of shape (num_perturbations,)
      all_distances: list of distance arrays for plotting
      d0_values: initial distances
      dN_values: final distances
    """
    # Initialize base weights
    np.random.seed(42)
    weights_base = init_weights(NUM_LAYERS)

    # Run base trajectory once
    traj_base = run_trajectory(weights_base, optimizer, lr, NUM_STEPS)

    lyapunov_exponents = []
    all_distances = []
    d0_values = []
    dN_values = []

    for p in range(num_perturbations):
        rng_pert = np.random.RandomState(seed_base + p)

        # Create perturbed initial weights
        weights_perturbed = []
        for layer_idx in range(NUM_LAYERS):
            W0 = weights_base[layer_idx].copy()

            if perturbation_type == 'gauge':
                # Gauge direction: W' = W @ (I + eps * S / ||S||_F)
                # S is symmetric => (I + eps*S) is approximately PSD for small eps
                S = random_symmetric(DIM, rng_pert)
                S = S / np.linalg.norm(S, 'fro')
                W_pert = W0 @ (np.eye(DIM) + EPSILON * S)
            elif perturbation_type == 'physical':
                # Physical/tangent direction: W' = W @ expm(eps * A / ||A||_F)
                # A is skew-symmetric => expm(A) is orthogonal
                A = random_skew_symmetric(DIM, rng_pert)
                A = A / np.linalg.norm(A, 'fro')
                R = matrix_exponential_pade(EPSILON * A)
                W_pert = W0 @ R
            else:
                raise ValueError(f"Unknown perturbation type: {perturbation_type}")

            weights_perturbed.append(W_pert)

        # Run perturbed trajectory
        traj_perturbed = run_trajectory(weights_perturbed, optimizer, lr, NUM_STEPS)

        # Compute distances
        distances = compute_trajectory_distance(traj_base, traj_perturbed)
        all_distances.append(distances)

        d0 = distances[0]
        dN = distances[-1]
        d0_values.append(d0)
        dN_values.append(dN)

        # Lyapunov exponent: lambda = (1/N) * log(d(N) / d(0))
        if d0 > 1e-15 and dN > 1e-15:
            lyap = (1.0 / NUM_STEPS) * np.log(dN / d0)
        elif dN < 1e-15:
            lyap = -np.inf  # Perturbation collapsed
        else:
            lyap = np.nan

        lyapunov_exponents.append(lyap)

    return (np.array(lyapunov_exponents), all_distances,
            np.array(d0_values), np.array(dN_values))

## 6. Main Experiment Execution

### Phase 1: Sanity Checks

Before measuring Lyapunov exponents, we verify that:
1. Both optimizers can train the network (loss decreases)
2. SGD's learning rate is stable (no divergence)
3. The initial loss is reasonable given our initialization

This is critical -- if an optimizer diverges, the Lyapunov exponent measurement is meaningless.

In [ ]:
print("=" * 100)
print("1.2b-i: LYAPUNOV EXPONENT OF GAUGE DIRECTION UNDER EACH OPTIMIZER")
print("=" * 100)
print(f"Setup: {NUM_LAYERS}-layer deep linear net (dim={DIM}), quadratic loss, {NUM_STEPS} steps")
print(f"Perturbation: eps={EPSILON}, {NUM_PERTURBATIONS} random directions")
print(f"LR_Muon={LR_MUON}, Momentum={MOMENTUM}")
print("=" * 100)

In [ ]:
# Find stable SGD learning rate
lr_sgd = find_stable_lr_sgd()
print(f"\nSGD learning rate (max stable): {lr_sgd}")
print(f"Muon learning rate (fixed):     {LR_MUON}")
print(f"LR ratio (Muon/SGD):            {LR_MUON / lr_sgd:.3f}")
print(f"  -> {'Muon uses a LARGER LR' if LR_MUON > lr_sgd else 'Muon uses a SMALLER LR' if LR_MUON < lr_sgd else 'Same LR'}")
print(f"  Note: This LR ratio is important context -- if Muon's LR is much larger,")
print(f"  it could confound the Lyapunov measurement. Watch for this.")

In [ ]:
# Verify both optimizers train properly
np.random.seed(42)
w_test = init_weights(NUM_LAYERS)
loss_init = compute_loss(w_test, X_data, W_target)
print(f"\nInitial loss: {loss_init:.6e}")

# Show the initial effective matrix
W_eff_init = np.eye(DIM)
for w in w_test:
    W_eff_init = w @ W_eff_init
print(f"Initial W_eff norm:       {np.linalg.norm(W_eff_init, 'fro'):.4f}")
print(f"Target W_target norm:     {np.linalg.norm(W_target, 'fro'):.4f}")
print(f"||W_eff - W_target||_F:   {np.linalg.norm(W_eff_init - W_target, 'fro'):.4f}")
print(f"  -> The network starts far from the target (loss is large).")

In [ ]:
# Quick check: run both optimizers and show training dynamics
print("Training validation (checking both optimizers converge):")
print("-" * 60)
for opt_name, opt_type, lr in [('SGD', 'sgd', lr_sgd), ('Muon', 'muon', LR_MUON)]:
    np.random.seed(42)
    w_check = init_weights(NUM_LAYERS)
    traj_check = run_trajectory(w_check, opt_type, lr, NUM_STEPS)
    loss_final = compute_loss(traj_check[-1], X_data, W_target)
    # Also check intermediate losses
    loss_25 = compute_loss(traj_check[50], X_data, W_target)
    loss_50 = compute_loss(traj_check[100], X_data, W_target)
    loss_75 = compute_loss(traj_check[150], X_data, W_target)
    reduction = loss_init / loss_final if loss_final > 0 else float('inf')
    print(f"  {opt_name} (lr={lr}):")
    print(f"    step   0: loss = {loss_init:.6e}")
    print(f"    step  50: loss = {loss_25:.6e}")
    print(f"    step 100: loss = {loss_50:.6e}")
    print(f"    step 150: loss = {loss_75:.6e}")
    print(f"    step 200: loss = {loss_final:.6e}")
    print(f"    Loss reduction factor: {reduction:.1f}x")
    print(f"    -> {'CONVERGING' if loss_final < loss_init else 'DIVERGING (PROBLEM!)'}")
    print()

### Phase 2: Lyapunov Exponent Measurement

Now we run the core measurement. For each of the 4 conditions (2 optimizers x 2 perturbation types), we:

1. Run 20 trials with different random perturbation directions
2. Compute the Lyapunov exponent for each trial
3. Aggregate statistics (mean, median, std)

**What to watch for in the output below:**
- **Mean lambda values:** The sign tells us stability. Negative = decaying perturbation = gauge-fixing.
- **d(N)/d(0) ratio:** Values < 1 mean the perturbation shrank; > 1 means it grew.
- **Comparison across conditions:** The key test is whether Muon's gauge lambda is significantly more negative than SGD's.

In [ ]:
print(f"\n{'=' * 100}")
print("MEASURING LYAPUNOV EXPONENTS")
print("=" * 100)

In [ ]:
results = {}

In [ ]:
for opt_name, opt_type, lr in [('SGD', 'sgd', lr_sgd), ('Muon', 'muon', LR_MUON)]:
    for pert_name, pert_type in [('gauge', 'gauge'), ('physical', 'physical')]:
        key = f"{opt_name}_{pert_name}"
        print(f"\n  Running {opt_name} with {pert_name} perturbation "
              f"({NUM_PERTURBATIONS} trials)...", flush=True)

        lyaps, dists, d0s, dNs = measure_lyapunov(
            opt_type, lr, pert_type, NUM_PERTURBATIONS
        )

        # Filter out any inf/nan
        valid_mask = np.isfinite(lyaps)
        lyaps_valid = lyaps[valid_mask]

        results[key] = {
            'lyapunov_all': lyaps,
            'lyapunov_valid': lyaps_valid,
            'distances': dists,
            'd0': d0s,
            'dN': dNs,
            'mean_lyap': np.mean(lyaps_valid) if len(lyaps_valid) > 0 else np.nan,
            'std_lyap': np.std(lyaps_valid) if len(lyaps_valid) > 0 else np.nan,
            'median_lyap': np.median(lyaps_valid) if len(lyaps_valid) > 0 else np.nan,
        }

        print(f"    Valid trials: {len(lyaps_valid)}/{NUM_PERTURBATIONS}")
        print(f"    Mean lambda:   {results[key]['mean_lyap']:.6f}")
        print(f"    Median lambda: {results[key]['median_lyap']:.6f}")
        print(f"    Std lambda:    {results[key]['std_lyap']:.6f}")
        print(f"    Mean d(0):     {np.mean(d0s):.6e}")
        print(f"    Mean d(N):     {np.mean(dNs):.6e}")
        print(f"    Ratio d(N)/d(0): {np.mean(dNs)/np.mean(d0s):.4f}")

        # Scientific interpretation of this condition
        ml = results[key]['mean_lyap']
        if ml < -0.01:
            interp = "STRONGLY STABLE -- perturbations decay exponentially fast"
        elif ml < -0.001:
            interp = "WEAKLY STABLE -- perturbations decay slowly"
        elif ml < 0.001:
            interp = "NEUTRAL -- perturbations neither grow nor decay"
        elif ml < 0.01:
            interp = "WEAKLY UNSTABLE -- perturbations grow slowly"
        else:
            interp = "STRONGLY UNSTABLE -- perturbations grow exponentially"
        print(f"    Interpretation: {interp}")
        # Show the implied half-life or doubling time
        if abs(ml) > 1e-6:
            tau = np.log(2) / abs(ml)
            if ml < 0:
                print(f"    Perturbation half-life: {tau:.1f} steps (decays to 50% in {tau:.1f} steps)")
            else:
                print(f"    Perturbation doubling time: {tau:.1f} steps (doubles every {tau:.1f} steps)")

## 7. Detailed Results

### Summary Table

The table below shows all four conditions side by side. The key column is **Sign**: DECAY means the optimizer is stabilizing that direction, GROW means it is destabilizing it, and NEUTRAL means it is indifferent.

**What to look for:**
- Muon + gauge should show DECAY (confirming gauge-fixing)
- SGD + gauge should show NEUTRAL or GROW (no gauge-fixing)
- Both optimizers + physical should behave similarly (physical directions are not gauge)

In [ ]:
print(f"\n\n{'=' * 100}")
print("DETAILED LYAPUNOV EXPONENT RESULTS")
print("=" * 100)

In [ ]:
print(f"\n{'Optimizer':<10} | {'Perturbation':<12} | {'Mean lambda':>12} | {'Median lambda':>14} | "
      f"{'Std':>8} | {'d(N)/d(0)':>10} | {'Sign':>8}")
print("-" * 90)

In [ ]:
for opt_name in ['SGD', 'Muon']:
    for pert_name in ['gauge', 'physical']:
        key = f"{opt_name}_{pert_name}"
        r = results[key]
        ratio = np.mean(r['dN']) / np.mean(r['d0']) if np.mean(r['d0']) > 0 else np.nan

        if r['mean_lyap'] < -0.001:
            sign_str = "DECAY"
        elif r['mean_lyap'] > 0.001:
            sign_str = "GROW"
        else:
            sign_str = "NEUTRAL"

        print(f"{opt_name:<10} | {pert_name:<12} | {r['mean_lyap']:12.6f} | "
              f"{r['median_lyap']:14.6f} | {r['std_lyap']:8.6f} | {ratio:10.4f} | {sign_str:>8}")

### Per-Trial Breakdown (Gauge Direction)

This table shows every individual trial's Lyapunov exponent in the gauge direction. Look for:
- **Consistency:** Are all trials for a given optimizer roughly the same sign? High variance would suggest the measurement is noisy.
- **Separation:** Is there clear separation between SGD and Muon columns? Overlapping distributions would weaken the conclusion.
- **Outliers:** Any trial with very different behavior from the rest may indicate a special perturbation direction worth investigating.

In [ ]:
print(f"\n\n{'=' * 100}")
print("PER-TRIAL LYAPUNOV EXPONENTS (GAUGE DIRECTION)")
print("=" * 100)

In [ ]:
print(f"\n{'Trial':>6} | {'SGD lambda':>12} | {'Muon lambda':>12} | {'SGD d(N)/d(0)':>14} | "
      f"{'Muon d(N)/d(0)':>14}")
print("-" * 75)

In [ ]:
sgd_g = results['SGD_gauge']
muon_g = results['Muon_gauge']

In [ ]:
for p in range(NUM_PERTURBATIONS):
    sgd_ratio = sgd_g['dN'][p] / sgd_g['d0'][p] if sgd_g['d0'][p] > 0 else np.nan
    muon_ratio = muon_g['dN'][p] / muon_g['d0'][p] if muon_g['d0'][p] > 0 else np.nan
    print(f"{p:6d} | {sgd_g['lyapunov_all'][p]:12.6f} | {muon_g['lyapunov_all'][p]:12.6f} | "
          f"{sgd_ratio:14.6f} | {muon_ratio:14.6f}")

### Per-Layer Distance Evolution

This analysis breaks down the trajectory distance by layer. In the RG gauge-fixing theory, the gauge-fixing effect should be present at **every layer** where Muon is applied, not just in aggregate.

**What to look for:**
- For Muon: all layers should show decaying distances (negative per-layer lambda)
- For SGD: layers may show neutral or growing distances
- The D-TEST found per-layer Lyapunov ~0.095 for SGD -- we should see similar or consistent values here
- If some layers decay and others grow, this reveals the spatial structure of gauge-fixing in the network

In [ ]:
print(f"\n\n{'=' * 100}")
print("PER-LAYER DISTANCE EVOLUTION (Trial 0, Gauge Perturbation)")
print("=" * 100)

In [ ]:
# Re-run trial 0 with per-layer tracking
np.random.seed(42)
weights_base = init_weights(NUM_LAYERS)
rng_layer = np.random.RandomState(100)

In [ ]:
# Create gauge-perturbed weights for trial 0
weights_pert_gauge = []
for layer_idx in range(NUM_LAYERS):
    S = random_symmetric(DIM, rng_layer)
    S = S / np.linalg.norm(S, 'fro')
    W_pert = weights_base[layer_idx] @ (np.eye(DIM) + EPSILON * S)
    weights_pert_gauge.append(W_pert)

In [ ]:
for opt_name, opt_type, lr in [('SGD', 'sgd', lr_sgd), ('Muon', 'muon', LR_MUON)]:
    traj_base = run_trajectory(weights_base, opt_type, lr, NUM_STEPS)
    traj_pert = run_trajectory(weights_pert_gauge, opt_type, lr, NUM_STEPS)
    per_layer = compute_per_layer_distances(traj_base, traj_pert)

    print(f"\n  {opt_name}:")
    print(f"  {'Layer':>6} | {'d(0)':>10} | {'d(50)':>10} | {'d(100)':>10} | "
          f"{'d(150)':>10} | {'d(200)':>10} | {'lambda_layer':>12}")
    print("  " + "-" * 80)

    for layer_idx in range(NUM_LAYERS):
        d0_l = per_layer[layer_idx, 0]
        dN_l = per_layer[layer_idx, -1]
        lyap_l = (1.0 / NUM_STEPS) * np.log(dN_l / d0_l) if d0_l > 1e-15 and dN_l > 1e-15 else np.nan
        print(f"  {layer_idx:6d} | {per_layer[layer_idx, 0]:10.6f} | "
              f"{per_layer[layer_idx, 50]:10.6f} | {per_layer[layer_idx, 100]:10.6f} | "
              f"{per_layer[layer_idx, 150]:10.6f} | {per_layer[layer_idx, -1]:10.6f} | "
              f"{lyap_l:12.6f}")

**Interpretation of per-layer results:** If Muon shows negative $\lambda_{\text{layer}}$ across all layers while SGD shows near-zero or positive values, this confirms that Muon's gauge-fixing is a *local* per-layer phenomenon driven by the Newton-Schulz step, not a global artifact of the loss landscape.

## 8. Statistical Significance Test

We perform a **Welch's t-test** (two-sample t-test with unequal variances) to determine whether the difference in gauge Lyapunov exponents between SGD and Muon is statistically significant.

- $H_0$: $\lambda_{\text{gauge}}^{\text{SGD}} = \lambda_{\text{gauge}}^{\text{Muon}}$ (both optimizers treat gauge directions equally)
- $H_1$: $\lambda_{\text{gauge}}^{\text{SGD}} > \lambda_{\text{gauge}}^{\text{Muon}}$ (Muon is more stable in gauge directions)

This is a one-tailed test because our theory makes a directional prediction. A t-statistic > 2.0 corresponds roughly to $p < 0.05$.

Note: We implement the t-test manually (no scipy dependency) using Welch's approximation for degrees of freedom.

In [ ]:
print(f"\n\n{'=' * 100}")
print("STATISTICAL SIGNIFICANCE TEST")
print("=" * 100)

In [ ]:
sgd_gauge_lyaps = results['SGD_gauge']['lyapunov_valid']
muon_gauge_lyaps = results['Muon_gauge']['lyapunov_valid']

In [ ]:
# Two-sample t-test (manual, no scipy)
n1, n2 = len(sgd_gauge_lyaps), len(muon_gauge_lyaps)
mean1, mean2 = np.mean(sgd_gauge_lyaps), np.mean(muon_gauge_lyaps)
var1, var2 = np.var(sgd_gauge_lyaps, ddof=1), np.var(muon_gauge_lyaps, ddof=1)

In [ ]:
if n1 > 1 and n2 > 1:
    se = np.sqrt(var1 / n1 + var2 / n2)
    if se > 1e-15:
        t_stat = (mean1 - mean2) / se
        # Welch's degrees of freedom
        df_num = (var1 / n1 + var2 / n2) ** 2
        df_den = (var1 / n1) ** 2 / (n1 - 1) + (var2 / n2) ** 2 / (n2 - 1)
        df = df_num / (df_den + 1e-15)
    else:
        t_stat = np.inf if mean1 != mean2 else 0.0
        df = min(n1, n2) - 1
else:
    t_stat = np.nan
    df = np.nan

In [ ]:
print(f"\n  H0: lambda_gauge_SGD = lambda_gauge_Muon")
print(f"  H1: lambda_gauge_SGD > lambda_gauge_Muon (Muon more stable)")
print(f"\n  SGD gauge Lyapunov:  mean={mean1:.6f}, std={np.sqrt(var1):.6f}, n={n1}")
print(f"  Muon gauge Lyapunov: mean={mean2:.6f}, std={np.sqrt(var2):.6f}, n={n2}")
print(f"  Difference (SGD - Muon): {mean1 - mean2:.6f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  Degrees of freedom: {df:.1f}")
print(f"  (t > 2.0 indicates statistical significance at p < 0.05, one-tailed)")

In [ ]:
is_significant = t_stat > 2.0 if np.isfinite(t_stat) else False
print(f"  Statistically significant: {'YES' if is_significant else 'NO'}")

**Statistical interpretation:** A statistically significant result means we can reject the null hypothesis that SGD and Muon have the same gauge Lyapunov exponent. Combined with the sign of the difference, this tells us whether Muon genuinely stabilizes gauge directions more than SGD, or whether the observed difference could plausibly arise from random variation in the perturbation directions.

## 9. Gauge vs. Physical Perturbation Comparison

This is the **specificity test**. The RG gauge-fixing hypothesis predicts that Muon specifically contracts gauge (symmetric/PSD) directions while being roughly neutral toward physical (skew-symmetric/tangent) directions.

If Muon simply contracts *all* perturbations (gauge and physical alike), then the effect is not gauge-fixing -- it is just generic contraction, possibly due to a smaller effective learning rate or stronger damping. Only if $\lambda_{\text{gauge}}^{\text{Muon}} \ll \lambda_{\text{physical}}^{\text{Muon}}$ can we conclude that Muon's Newton-Schulz step specifically targets the gauge redundancy.

**This is the most discriminating test in the experiment.**

In [ ]:
print(f"\n\n{'=' * 100}")
print("GAUGE vs PHYSICAL PERTURBATION COMPARISON")
print("=" * 100)

In [ ]:
for opt_name in ['SGD', 'Muon']:
    gauge_key = f"{opt_name}_gauge"
    phys_key = f"{opt_name}_physical"
    print(f"\n  {opt_name}:")
    print(f"    Gauge lambda:    {results[gauge_key]['mean_lyap']:.6f} +/- {results[gauge_key]['std_lyap']:.6f}")
    print(f"    Physical lambda: {results[phys_key]['mean_lyap']:.6f} +/- {results[phys_key]['std_lyap']:.6f}")
    diff = results[gauge_key]['mean_lyap'] - results[phys_key]['mean_lyap']
    print(f"    Difference (gauge - physical): {diff:.6f}")
    if abs(diff) > 0.001:
        if diff < 0:
            print(f"    -> Gauge direction is MORE STABLE than physical (confirms gauge fixing)")
        else:
            print(f"    -> Physical direction is MORE STABLE than gauge (unexpected)")
    else:
        print(f"    -> Similar stability in both directions")

## 10. Visualization

### Plot 1: Distance Evolution $d(t)$ Over Time (4-Panel Overview)

This is the primary visualization. Four panels show:

- **(a) Gauge perturbation, d(t) over time:** The most important panel. If Muon is gauge-fixing, the red curves (Muon) should slope downward on the log scale, while blue curves (SGD) remain flat or slope upward.
- **(b) Physical perturbation, d(t) over time:** Control condition. Both optimizers should behave similarly here.
- **(c) Histogram of Lyapunov exponents (gauge):** Shows the distribution across trials. Non-overlapping distributions indicate a robust effect.
- **(d) Bar chart comparison:** Summary of all four conditions with error bars.

**Visual signatures of gauge-fixing:**
- Muon's gauge curves drop below SGD's on log scale
- Muon's gauge histogram is shifted left (more negative) relative to SGD's
- Muon's gauge bar is more negative than its physical bar

In [ ]:
print(f"\n\n{'=' * 100}")
print("GENERATING PLOTS")
print("=" * 100)

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('1.2b-i: Lyapunov Exponent of Gauge Direction\n'
                 f'{NUM_LAYERS}-layer linear net, dim={DIM}, eps={EPSILON}, '
                 f'{NUM_PERTURBATIONS} perturbations',
                 fontsize=14, fontweight='bold')

    t_axis = np.arange(NUM_STEPS + 1)

    # --- Panel (a): d(t) for gauge perturbation, both optimizers ---
    ax = axes[0, 0]
    ax.set_title('(a) Gauge Perturbation: d(t) over time')

    # Plot individual trials (faint) and mean (bold)
    for p in range(NUM_PERTURBATIONS):
        dists_sgd = results['SGD_gauge']['distances'][p]
        dists_muon = results['Muon_gauge']['distances'][p]
        ax.semilogy(t_axis[:len(dists_sgd)], dists_sgd, 'b-', alpha=0.1, linewidth=0.5)
        ax.semilogy(t_axis[:len(dists_muon)], dists_muon, 'r-', alpha=0.1, linewidth=0.5)

    # Mean distance
    sgd_mean_dist = np.mean([d for d in results['SGD_gauge']['distances']], axis=0)
    muon_mean_dist = np.mean([d for d in results['Muon_gauge']['distances']], axis=0)
    ax.semilogy(t_axis[:len(sgd_mean_dist)], sgd_mean_dist, 'b-', linewidth=2.5,
                label=f'SGD (lambda={results["SGD_gauge"]["mean_lyap"]:.4f})')
    ax.semilogy(t_axis[:len(muon_mean_dist)], muon_mean_dist, 'r-', linewidth=2.5,
                label=f'Muon (lambda={results["Muon_gauge"]["mean_lyap"]:.4f})')

    # Reference: d(0) line
    ax.axhline(y=np.mean(results['SGD_gauge']['d0']), color='gray', linestyle='--',
               alpha=0.5, label='d(0)')
    ax.set_xlabel('Step')
    ax.set_ylabel('d(t) = ||W_t\' - W_t||_F')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Panel (b): d(t) for physical perturbation, both optimizers ---
    ax = axes[0, 1]
    ax.set_title('(b) Physical Perturbation: d(t) over time')

    for p in range(NUM_PERTURBATIONS):
        dists_sgd = results['SGD_physical']['distances'][p]
        dists_muon = results['Muon_physical']['distances'][p]
        ax.semilogy(t_axis[:len(dists_sgd)], dists_sgd, 'b-', alpha=0.1, linewidth=0.5)
        ax.semilogy(t_axis[:len(dists_muon)], dists_muon, 'r-', alpha=0.1, linewidth=0.5)

    sgd_phys_mean = np.mean([d for d in results['SGD_physical']['distances']], axis=0)
    muon_phys_mean = np.mean([d for d in results['Muon_physical']['distances']], axis=0)
    ax.semilogy(t_axis[:len(sgd_phys_mean)], sgd_phys_mean, 'b-', linewidth=2.5,
                label=f'SGD (lambda={results["SGD_physical"]["mean_lyap"]:.4f})')
    ax.semilogy(t_axis[:len(muon_phys_mean)], muon_phys_mean, 'r-', linewidth=2.5,
                label=f'Muon (lambda={results["Muon_physical"]["mean_lyap"]:.4f})')
    ax.axhline(y=np.mean(results['SGD_physical']['d0']), color='gray', linestyle='--',
               alpha=0.5, label='d(0)')
    ax.set_xlabel('Step')
    ax.set_ylabel('d(t) = ||W_t\' - W_t||_F')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Panel (c): Histogram of Lyapunov exponents ---
    ax = axes[1, 0]
    ax.set_title('(c) Distribution of Lyapunov Exponents (Gauge)')

    bins = np.linspace(
        min(np.min(results['SGD_gauge']['lyapunov_valid']),
            np.min(results['Muon_gauge']['lyapunov_valid'])) - 0.005,
        max(np.max(results['SGD_gauge']['lyapunov_valid']),
            np.max(results['Muon_gauge']['lyapunov_valid'])) + 0.005,
        25
    )
    ax.hist(results['SGD_gauge']['lyapunov_valid'], bins=bins, alpha=0.6, color='blue',
            label=f'SGD (mean={results["SGD_gauge"]["mean_lyap"]:.4f})', edgecolor='navy')
    ax.hist(results['Muon_gauge']['lyapunov_valid'], bins=bins, alpha=0.6, color='red',
            label=f'Muon (mean={results["Muon_gauge"]["mean_lyap"]:.4f})', edgecolor='darkred')
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='lambda=0 (neutral)')
    ax.set_xlabel('Lyapunov Exponent (lambda)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Panel (d): Gauge vs Physical comparison (bar chart) ---
    ax = axes[1, 1]
    ax.set_title('(d) Gauge vs Physical Lyapunov Exponents')

    categories = ['SGD\nGauge', 'SGD\nPhysical', 'Muon\nGauge', 'Muon\nPhysical']
    means = [
        results['SGD_gauge']['mean_lyap'],
        results['SGD_physical']['mean_lyap'],
        results['Muon_gauge']['mean_lyap'],
        results['Muon_physical']['mean_lyap'],
    ]
    stds = [
        results['SGD_gauge']['std_lyap'],
        results['SGD_physical']['std_lyap'],
        results['Muon_gauge']['std_lyap'],
        results['Muon_physical']['std_lyap'],
    ]
    colors = ['#4477AA', '#88CCEE', '#CC3311', '#EE7733']

    bars = ax.bar(categories, means, yerr=stds, capsize=5, color=colors,
                  edgecolor='black', linewidth=0.8)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5)
    ax.set_ylabel('Lyapunov Exponent (lambda)')

    # Annotate bars
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                f'{mean:.4f}', ha='center', va='bottom' if mean >= 0 else 'top',
                fontsize=9, fontweight='bold')

    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plot_path = os.path.join(SCRIPT_DIR, 'lyapunov_gauge_direction.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n  Plot saved to: {plot_path}")

In [ ]:
except ImportError:
    print("\n  WARNING: matplotlib not available, skipping plots.")
    plot_path = None

### Plot 2: $\log(d(t)/d(0))$ vs. $t$ -- Direct Lyapunov Visualization

This plot makes the Lyapunov exponent directly visible as the **slope** of the curve. A straight line with slope $\lambda$ means $d(t) = d(0) e^{\lambda t}$, i.e., perfect exponential growth/decay.

- **Negative slope** = stable (perturbation decays)
- **Zero slope** = neutral (perturbation constant)
- **Positive slope** = unstable (perturbation grows)

The dashed lines show the linear fit $\lambda \cdot t$ for comparison. Deviation from linearity indicates non-exponential behavior (e.g., power-law decay or saturation effects).

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('1.2b-i: log(d(t)/d(0)) vs t -- Slope = Lyapunov Exponent',
                 fontsize=13, fontweight='bold')

    for idx, (pert_name, pert_label) in enumerate([('gauge', 'Gauge (Symmetric/PSD)'),
                                                     ('physical', 'Physical (Skew/Tangent)')]):
        ax = axes[idx]
        ax.set_title(pert_label)

        for opt_name, color in [('SGD', 'blue'), ('Muon', 'red')]:
            key = f"{opt_name}_{pert_name}"
            dists_list = results[key]['distances']

            # Plot individual trials
            for p in range(NUM_PERTURBATIONS):
                d = dists_list[p]
                d0 = d[0]
                if d0 > 1e-15:
                    log_ratio = np.log(d / d0)
                    ax.plot(t_axis[:len(log_ratio)], log_ratio, color=color,
                            alpha=0.1, linewidth=0.5)

            # Mean
            mean_dist = np.mean(dists_list, axis=0)
            d0_mean = mean_dist[0]
            if d0_mean > 1e-15:
                log_ratio_mean = np.log(mean_dist / d0_mean)
                ax.plot(t_axis[:len(log_ratio_mean)], log_ratio_mean, color=color,
                        linewidth=2.5, label=f'{opt_name} (lambda={results[key]["mean_lyap"]:.4f})')

                # Linear fit line
                lyap = results[key]['mean_lyap']
                ax.plot(t_axis, lyap * t_axis, color=color, linestyle='--',
                        linewidth=1.5, alpha=0.7)

        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax.set_xlabel('Step')
        ax.set_ylabel('log(d(t) / d(0))')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path2 = os.path.join(SCRIPT_DIR, 'lyapunov_log_ratio.png')
    plt.savefig(plot_path2, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Plot saved to: {plot_path2}")

In [ ]:
except ImportError:
    pass

## 11. Final Verdict

### Three Hypotheses Tested

We evaluate the experiment against three increasingly stringent predictions of the RG gauge-fixing theory:

1. **H1 (Comparative):** $\lambda_{\text{gauge}}^{\text{Muon}} < \lambda_{\text{gauge}}^{\text{SGD}}$ -- Muon is more stable than SGD in the gauge direction. This is the weakest test; it could be satisfied even if both are negative.

2. **H2 (Absolute):** $\lambda_{\text{gauge}}^{\text{Muon}} \ll 0$ -- Muon's gauge Lyapunov exponent is strongly negative. This confirms that gauge perturbations *decay* under Muon, not just that Muon is "less unstable" than SGD.

3. **H3 (Specificity):** $\lambda_{\text{gauge}}^{\text{Muon}} < \lambda_{\text{physical}}^{\text{Muon}}$ -- Muon is more stable in the gauge direction than in the physical direction. This is the strongest test, confirming that Muon *specifically* targets gauge directions rather than just contracting all perturbations.

In [ ]:
print(f"\n\n{'=' * 100}")
print("FINAL VERDICT: LYAPUNOV EXPONENT OF GAUGE DIRECTION")
print("=" * 100)

In [ ]:
lambda_gauge_sgd = results['SGD_gauge']['mean_lyap']
lambda_gauge_muon = results['Muon_gauge']['mean_lyap']
lambda_phys_sgd = results['SGD_physical']['mean_lyap']
lambda_phys_muon = results['Muon_physical']['mean_lyap']

In [ ]:
print(f"""
  MEASURED LYAPUNOV EXPONENTS:
  ---------------------------------------------------------------
  lambda_gauge_SGD    = {lambda_gauge_sgd:+.6f}  ({"DECAY" if lambda_gauge_sgd < -0.001 else "GROW" if lambda_gauge_sgd > 0.001 else "NEUTRAL"})
  lambda_gauge_Muon   = {lambda_gauge_muon:+.6f}  ({"DECAY" if lambda_gauge_muon < -0.001 else "GROW" if lambda_gauge_muon > 0.001 else "NEUTRAL"})
  lambda_phys_SGD     = {lambda_phys_sgd:+.6f}  ({"DECAY" if lambda_phys_sgd < -0.001 else "GROW" if lambda_phys_sgd > 0.001 else "NEUTRAL"})
  lambda_phys_Muon    = {lambda_phys_muon:+.6f}  ({"DECAY" if lambda_phys_muon < -0.001 else "GROW" if lambda_phys_muon > 0.001 else "NEUTRAL"})
  ---------------------------------------------------------------

  HYPOTHESIS CHECK:
  ---------------------------------------------------------------
  H1: lambda_gauge_Muon < lambda_gauge_SGD
      (Muon stabilizes gauge directions more than SGD)
      Muon: {lambda_gauge_muon:+.6f}  vs  SGD: {lambda_gauge_sgd:+.6f}
      Difference: {lambda_gauge_sgd - lambda_gauge_muon:.6f}
      -> {"CONFIRMED" if lambda_gauge_muon < lambda_gauge_sgd else "REJECTED"}
      {"   (statistically significant)" if is_significant else "   (NOT statistically significant)"}

  H2: lambda_gauge_Muon << 0 (strongly negative)
      -> {"CONFIRMED" if lambda_gauge_muon < -0.01 else "PARTIALLY CONFIRMED" if lambda_gauge_muon < -0.001 else "REJECTED"}

  H3: lambda_gauge_SGD ~ 0 (neutral) or > 0 (unstable)
      -> {"CONFIRMED (neutral)" if abs(lambda_gauge_sgd) < 0.005 else "CONFIRMED (unstable)" if lambda_gauge_sgd > 0.005 else "REJECTED (SGD also decays gauge)"}
  ---------------------------------------------------------------
""")

In [ ]:
# Determine overall pass/fail
tests_passed = 0
tests_total = 3

In [ ]:
# Test 1: lambda_gauge_Muon < lambda_gauge_SGD
test1_pass = lambda_gauge_muon < lambda_gauge_sgd
if test1_pass:
    tests_passed += 1

In [ ]:
# Test 2: lambda_gauge_Muon < 0 (Muon decays gauge perturbations)
test2_pass = lambda_gauge_muon < -0.001
if test2_pass:
    tests_passed += 1

In [ ]:
# Test 3: Muon's gauge Lyapunov is more negative than its physical Lyapunov
# (Muon specifically targets gauge directions, not just all directions)
test3_pass = lambda_gauge_muon < lambda_phys_muon - 0.001
if test3_pass:
    tests_passed += 1

In [ ]:
if tests_passed == 3:
    overall = "PASS"
    detail = (
        "All three tests pass:\n"
        "  1. Muon's gauge Lyapunov < SGD's gauge Lyapunov (Muon more stable)\n"
        "  2. Muon's gauge Lyapunov < 0 (perturbations decay)\n"
        "  3. Muon's gauge Lyapunov < Muon's physical Lyapunov\n"
        "     (Muon SPECIFICALLY stabilizes gauge directions)\n"
        "\n"
        "  This confirms the dynamical systems prediction: Muon acts as a\n"
        "  gauge-fixing mechanism that exponentially suppresses PSD perturbations."
    )
elif tests_passed >= 2:
    overall = "PARTIAL PASS"
    detail = (
        f"  {tests_passed}/3 tests pass.\n"
        f"  Test 1 (Muon < SGD):          {'PASS' if test1_pass else 'FAIL'}\n"
        f"  Test 2 (Muon gauge < 0):      {'PASS' if test2_pass else 'FAIL'}\n"
        f"  Test 3 (gauge < physical):     {'PASS' if test3_pass else 'FAIL'}\n"
        "\n"
        "  The core prediction is partially supported."
    )
elif tests_passed == 1:
    overall = "WEAK SIGNAL"
    detail = (
        f"  Only {tests_passed}/3 tests pass.\n"
        f"  Test 1 (Muon < SGD):          {'PASS' if test1_pass else 'FAIL'}\n"
        f"  Test 2 (Muon gauge < 0):      {'PASS' if test2_pass else 'FAIL'}\n"
        f"  Test 3 (gauge < physical):     {'PASS' if test3_pass else 'FAIL'}"
    )
else:
    overall = "FAIL"
    detail = (
        "  No tests pass. The Lyapunov exponent predictions are not confirmed.\n"
        f"  Test 1 (Muon < SGD):          {'PASS' if test1_pass else 'FAIL'}\n"
        f"  Test 2 (Muon gauge < 0):      {'PASS' if test2_pass else 'FAIL'}\n"
        f"  Test 3 (gauge < physical):     {'PASS' if test3_pass else 'FAIL'}"
    )

In [ ]:
print(f"""
  ╔══════════════════════════════════════════════════════════════════════════╗
  ║  VERDICT: {overall:<63}║
  ╠══════════════════════════════════════════════════════════════════════════╣
  ║                                                                        ║""")
for line in detail.split('\n'):
    print(f"  ║  {line:<70}║")
print(f"""  ║                                                                        ║
  ╚══════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
print("=" * 100)
print(f"  Tests passed: {tests_passed}/{tests_total}")
print(f"  Overall: {overall}")
print("=" * 100)

## Conclusions

### What This Experiment Measured

This experiment measured the **Lyapunov exponent** of optimization trajectories under gauge (symmetric/PSD) and physical (skew-symmetric/tangent) perturbations for both SGD and Muon optimizers. The Lyapunov exponent $\lambda$ quantifies whether small perturbations in a given direction grow ($\lambda > 0$), decay ($\lambda < 0$), or remain constant ($\lambda \approx 0$) over the course of optimization.

### Connection to the RG Gauge-Fixing Theory

In the framework of Muon as an RG gauge-fixing mechanism:

- **Gauge directions** (symmetric/PSD perturbations) represent the redundant degrees of freedom in deep linear networks -- transformations that change the individual layer matrices without changing the effective end-to-end linear map $W_{\text{eff}} = W_L \cdots W_1$.
- **Physical directions** (skew-symmetric/tangent perturbations) represent genuine changes to the network's function.
- A **gauge-fixing optimizer** should have $\lambda_{\text{gauge}} < 0$ (actively contracting gauge redundancies) while leaving $\lambda_{\text{physical}}$ approximately neutral.

The Newton-Schulz orthogonalization in Muon replaces each gradient with its orthogonal polar factor, which by construction has uniform singular values. This removes the symmetric (gauge) component of the gradient update, providing the theoretical mechanism for gauge-fixing.

### Key Results

The three hypothesis tests evaluated increasingly stringent predictions:

1. **H1 (Comparative):** Is $\lambda_{\text{gauge}}^{\text{Muon}} < \lambda_{\text{gauge}}^{\text{SGD}}$? -- Does Muon stabilize gauge directions more than SGD?
2. **H2 (Absolute):** Is $\lambda_{\text{gauge}}^{\text{Muon}} \ll 0$? -- Does Muon actively contract gauge perturbations?
3. **H3 (Specificity):** Is $\lambda_{\text{gauge}}^{\text{Muon}} < \lambda_{\text{physical}}^{\text{Muon}}$? -- Does Muon specifically target gauge directions, rather than contracting all perturbations generically?

### Implications

- If all three tests pass (**PASS**): Strong evidence that Muon acts as a gauge-fixing mechanism. The Newton-Schulz step specifically and exponentially suppresses PSD perturbations, consistent with the RG gauge-fixing interpretation.
- If H1 and H2 pass but H3 fails (**PARTIAL PASS**): Muon contracts gauge perturbations, but not specifically -- it may simply be a stronger optimizer that contracts all directions. The gauge-fixing interpretation is weakened.
- If tests mostly fail (**FAIL**): The Lyapunov exponent predictions of the RG gauge-fixing theory are not confirmed in this setting. This would motivate re-examining the theory's assumptions or testing in different architectures.

### Limitations and Future Directions

- **Deep linear networks only:** The gauge structure is exact for linear networks but approximate for nonlinear networks. Future experiments should test with ReLU or other activations.
- **Fixed architecture:** The 4-layer, 32x32 setup is a specific point in architecture space. Scaling to larger dimensions and depths would test robustness.
- **Single learning rate pair:** The LR calibration ensures fairness, but a systematic sweep over learning rates would strengthen the conclusion.
- **Finite-time Lyapunov:** The measured $\lambda$ is a finite-time estimate over $N = 200$ steps. The true asymptotic Lyapunov exponent may differ, especially if the trajectory passes through different regimes.

### Connection to Other Experiments

- **Exp 1.1 (D-TEST):** Established that SGD's gauge-direction disadvantage compounds exponentially with depth. This experiment measures the *rate* of that compounding.
- **Exp 1.2a (Spectrum Perturbation):** Tests whether Muon's singular value spectrum is invariant under gauge perturbations (a static test). This experiment is the *dynamic* counterpart.
- **Exp 1.2b-ii (Trajectory Divergence):** Will examine whether the Lyapunov effect translates into measurable trajectory divergence in weight space over longer horizons.

---

*Experiment 1.2b-i complete.*